<a href="https://colab.research.google.com/github/ShiX27/capstone_acc/blob/main/notebooks/scratch/2026-11-18%20%E2%80%94%20Capstone%20Kickoff%20and%20First%20Pulls%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Capstone Kickoff and First Pulls

**Studio — 2026-11-18 · Fall 2026**  
**Class time:** 45 minutes

---

## Capstone kickoff and first pulls

The capstone starts today. Read `DS2002 Capstone — ACC Reality Check.pdf` in this folder before Friday, because it describes everything your team will hand in and when. The short version is that your team will decide which ACC men's basketball teams are as good as their records say and which ones have built their records on easy opponents. To do that you will combine a ratings file with game results from a live API, keep everything in a SQLite database, answer the questions with SQL, and publish your results as a small web API.

Today has a modest goal. Each team forms, sets up a repository, and looks at both sources with its own eyes. You will not clean or load anything yet. The point of the hour is to find out what the data looks like before anyone writes a plan around it.

If you do not follow basketball, start with the appendix of the PDF. It explains every term used in this notebook, and nobody on your team should feel behind because of the sport.

### Step 1: Form your team

Teams have three or four people. Fill in the cell below together and commit it to the team repository by the end of class. The roles are a starting point, and you can trade them later, but someone should own each part of the project from the first day.

In [1]:
team = {
    'name': 'Capstone Project Team',
    'members': ['Simi', 'Xiangyu', 'Eyram'],
    'repo_url': 'https://github.com/ShiX27/capstone_acc',
    'roles': {
        'ratings CSV and crosswalk': 'Xiangyu',
        'ESPN pulls and box scores': 'Eyram',
        'SQLite schema and queries': 'Simi',
        'API and PythonAnywhere': 'Simi',
    },
    'meeting_time': 'Monday, Nov 23rd at 10:00AM',
}

unfilled = [k for k, v in team.items() if 'TODO' in str(v)]
print('still unfilled:', unfilled if unfilled else 'nothing, the charter is complete')

still unfilled: nothing, the charter is complete


### Step 2: Pull the ratings CSV

Bart Torvik publishes a rating for every Division I team as a CSV file at this address:

`https://barttorvik.com/2027_team_results.csv`

The 2027 in the name refers to the season that ends in 2027, which is the season being played right now. You do not need a key or an account.

**TODO:** download the file with `requests`, read it into pandas, and look at it before you do anything else. Print the shape, the column names, and the first few rows. Save the raw text to a file named with today's date, because the ratings change every day and you will want to know which version you used.

In [2]:
import requests
import pandas as pd
from io import StringIO
from pathlib import Path
from datetime import date

In [3]:
# TODO: request the CSV, save the raw text with today's date in the file name,
#       then read it into a DataFrame and look at it

torvik_2026_11_21 = pd.read_csv("torvik_2026-11-21.csv")
torvik_2026_11_21.head()

,rank,team,conf,record,adjoe,oe Rank,adjde,de Rank,barthag,rank.1,...,ConPA,ConPoss,ConOE,ConDE,ConSOSRemain,Conf Win%,WAB,WAB Rk,Fun Rk,adjt
0,1,Houston,B12,6-0,124.433108,3,93.529425,3,0.963847,1,...,0.0,0.0,0,0,0.839675,0,2.449693,1,28,67.794672
1,2,Illinois,B10,5-1,126.327737,1,96.453853,16,0.957011,2,...,0.0,0.0,0,0,0.835775,0,0.997270,19,129,73.811227
2,3,Kansas,B12,5-1,122.696551,6,93.999293,4,0.955379,3,...,0.0,0.0,0,0,0.834088,0,1.334751,7,159,73.019103
3,4,Duke,ACC,3-2,120.560558,14,92.630325,1,0.953940,4,...,0.0,0.0,0,0,0.810607,0,-0.465137,110,355,71.508852
4,5,Florida,SEC,6-1,124.715018,2,96.172924,14,0.952059,5,...,0.0,0.0,0,0,0.857882,0,2.295678,2,69,68.731843


In [4]:
print('Shape:', torvik_2026_11_21.shape)
print('Columns:', torvik_2026_11_21.columns.tolist())

Shape: (365, 45)
Columns: ['rank', 'team', 'conf', 'record', 'adjoe', 'oe Rank', 'adjde', 'de Rank', 'barthag', 'rank.1', 'proj. W', 'Proj. L', 'Pro Con W', 'Pro Con L', 'Con Rec.', 'sos', 'ncsos', 'consos', 'Proj. SOS', 'Proj. Noncon SOS', 'Proj. Con SOS', 'elite SOS', 'elite noncon SOS', 'Opp OE', 'Opp DE', 'Opp Proj. OE', 'Opp Proj DE', 'Con Adj OE', 'Con Adj DE', 'Qual O', 'Qual D', 'Qual Barthag', 'Qual Games', 'FUN', 'ConPF', 'ConPA', 'ConPoss', 'ConOE', 'ConDE', 'ConSOSRemain', 'Conf Win%', 'WAB', 'WAB Rk', 'Fun Rk', 'adjt']


**TODO:** answer these in the cell below, using what you printed.

1. How many rows are there, and how many belong to the ACC? You should be able to filter on one column to find out.
2. Two of the original headers have the same name. What did pandas call them, and which one would you trust as the team's rank?
3. Look at how a few schools are spelled, such as North Carolina State and Miami. Write the spellings down exactly, because the other source will spell them differently.

**1. There are 365 rows, of which 18 belong to the ACC, based on filtering the `conf` column.**

**2. Pandas named the two columns `rank` and `rank.1`. I would trust `rank.1` because it represents each team's national ranking based on Barthag, as described in Appendix B.**

**3. The Torvik dataset spells North Carolina State as `N.C. State`, Miami as `Miami FL`, and Florida State as `Florida St.`. These spellings are important to record because ESPN writes the same schools differently.**

### Step 3: Pull one day of games from ESPN

ESPN serves game results as JSON with no key. This address returns every game an ACC team played on one date:

`https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard`

Pass the parameters `groups=2` (the ACC), `dates=YYYYMMDD`, and `limit=200`. Pick a date from the past week. You practiced this kind of request in October, so reuse what worked for you then, including a timeout and a check of the status code.

**TODO:** make the request, then explore the JSON one level at a time. Find the list of games, count them, and for one game print both team names, both scores, whether the game is finished, and the start time exactly as ESPN wrote it.

In [5]:
# TODO: request one date, then find the games inside the JSON

import requests

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"

params = {
    "groups": 2,
    "dates": "20261114",
    "limit": 200
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()    # Covert JSON to Python

print("Status code:", response.status_code)

Status code: 200


In [6]:
# Explore the JSON structure
print(data.keys())

# Find and count the games
games = data["events"]

print("Number of games:", len(games))
for game in games:
    print(game['name'])

dict_keys(['leagues', 'groups', 'events', 'provider', 'eventsDate'])
Number of games: 3
UNC Asheville Bulldogs at Georgia Tech Yellow Jackets
Southern Utah Thunderbirds at Wake Forest Demon Deacons
Utah Tech Trailblazers at California Golden Bears


In [7]:
# Explore one game's JSON structure
game = games[0]
print(game.keys())

competition = game["competitions"][0]
print(competition.keys())

dict_keys(['id', 'uid', 'date', 'name', 'shortName', 'season', 'competitions', 'links', 'status'])
dict_keys(['id', 'uid', 'date', 'attendance', 'type', 'timeValid', 'neutralSite', 'conferenceCompetition', 'playByPlayAvailable', 'recent', 'venue', 'competitors', 'notes', 'status', 'broadcasts', 'format', 'startDate', 'broadcast', 'geoBroadcasts', 'headlines', 'highlights'])


In [8]:
# print both teams and their scores
for team in competition["competitors"]:
    print("Team:", team["team"]["location"])
    print("Score:", team["score"])

Team: Georgia Tech
Score: 75
Team: UNC Asheville
Score: 74


In [9]:
# print completion status and original start time
print("Game finished:", competition["status"]["type"]["completed"])
print("Start time:", competition["date"])

Game finished: True
Start time: 2026-11-14T21:00Z


**TODO:** answer these in the cell below.

1. Where in the JSON are the games, and where inside one game are the two teams?
2. Compare the start time ESPN gives for an evening game with the date you asked for. What time zone is the start time in, and what goes wrong if you use its first ten characters as the game date?
3. How does ESPN spell the schools you wrote down in Step 2?
4. Did every opponent on your date play in Division I? Look the unfamiliar names up.

**1. The games are stored in the `events` key of the JSON. Each game has two participating teams, which are stored in `competitions[0]["competitors"]`. Each competitor contains the team's name and score.**

**2. ESPN provides the game start times in UTC using an ISO-formatted timestamp, which is indivated by the Z at the end of the printed start time. This brings up a potential issues that if an evening game takes place in EST, its UTC timestamp could be represented on a different day. This is why using the first ten characters of the timestamp could incorrectly assign the game to the next day.**

**3. The Torvik dataset spells North Carolina State as `N.C. State`, Miami as `Miami FL`, and Florida State as `Florida St`. However, ESPN spells North Carolina State as `NC State`, Miami as `Miami`, and Florida State as `Florida State`. The differences will need to be corrected before we merge the datasets.**

**4. All 3 opponents on Nov 14th were Division I programs.**

### Step 4: Write down the work ahead

You have now seen both sources. Before you leave, list at least five specific things that will have to be cleaned, converted, or matched before the two sources can be joined, and make each one concrete. "Names are different" is less useful than "ESPN writes NC State and Torvik writes N.C. State." This list becomes the first entries in your decision log on Friday.

_1._ **We will need to standardize the team names between Torvik and ESPN, so that the same schools can be matched across datasets. For example, Torvik lists Florida State as `Florida St.`, while ESPN lists it as `Florida State`.**

_2._ **We will need to convert game dates and times. ESPN provides game start times in UTC, so we will need to account for time zone differences when assigning games to their correct dates.**

_3._ **We will need to convert data types, as ESPN stores scores as values that may need to become numeric data so we can perform analyses with pandas and SQL.**

_4._ **We will need to identify completed games and distinguish them from games that are still in progress or postponed.**

_5._ **We will need to confirm conference and opponent information, as ESPN includes ACC teams that play against schools outside of the ACC. This needs to be confirmed so we can keep the correct information and determine how to match schools to rating in the Torvik dataset.**

### Before Friday

Friday's check-in notebook walks your team through getting the first version of the pipeline running, including the name crosswalk and the first tables in SQLite. Read the PDF tonight, and agree as a team on who is starting which part before you leave the room.

### Team progress

This cell is for your team, and nothing is submitted or graded. The only graded work in the capstone is the final submission on Friday, December 18, so these progress cells exist to help you notice early when something is behind. Fill it in and commit it with the rest of the notebook.

In [10]:
progress = {
    'acc_rows_in_torvik': 18,     # TODO: how many ACC rows the CSV has
    'espn_date_pulled': '2026-11-14',     # TODO: the date you asked ESPN for
    'espn_games_that_day': 3,    # TODO: how many games came back
    'name_that_differs': 'North Carolina State',    # TODO: one school spelled two ways
    'blocker': 'nothing yet',              # TODO: what is in your way, or 'nothing yet'
}

for k, v in progress.items():
    print(f'{k}: {v}')

acc_rows_in_torvik: 18
espn_date_pulled: 2026-11-14
espn_games_that_day: 3
name_that_differs: North Carolina State
blocker: nothing yet
